## Setup 1

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model by using a `while True` loop and checking whether the latest response contains any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If the model returns only a normal `message` and no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps calling the model until the model stops asking for tools.


In [4]:
# install opentelemtry
# !uv add opentelemetry-api opentelemetry-sdk

OpenTelemetry

Before we start, we need to learn a few concepts from OTel - we will use them in this homework.

- A trace is the end-to-end story of a single request as it moves through your system. For us, it's one RAG call.
- A span is one operation within a trace. A trace is made of one or more spans, organized as a tree. Each span has a name, a start and end time, and a set of attributes. For us we will have one span inside the trace, but for agents one trace will have multiple spans.
- Attributes are key-value pairs attached to a span - anything you want to record, like the number of tokens used or the cost of a call.

When a span finishes - meaning the code block it wraps completes - the SDK hands it to a span processor, which forwards it to an exporter. The exporter decides where the span goes: to the console, to a file, to a database, or to a remote collector. We will see all of this in practice in the questions below.

We start with the ConsoleSpanExporter, which prints each finished span to the terminal so we can see what OTel captures:

In [6]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Here is what each line does:

- TracerProvider() creates the SDK's central configuration object. It owns the span processors and decides how spans are built.
- SimpleSpanProcessor(ConsoleSpanExporter()) wires a processor that forwards every finished span to the console exporter, one at a time. "Simple" means synchronous and immediate - good for development.
- trace.set_tracer_provider(provider) registers the provider globally, so every call to trace.get_tracer(...) returns a tracer backed by it.
- trace.get_tracer("llm-zoomcamp") returns a Tracer we use to create spans. The string is just a label for the instrumentation scope - it identifies which part of the code produced the spans.

Put this block at the top of your script, before you import or use starter - so the tracer provider is ready before any code that might create spans.

With the tracer in hand, you can wrap any block of code in a span:

```python
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")
```

start_as_current_span creates a new span and makes it the "current" span for the duration of the with block. Any code inside the block - including other calls to start_as_current_span - becomes a child of this span. When the block exits, the span ends automatically.

You will use this pattern to instrument the RAG methods in the questions below.

In [7]:
# try an llm call with a tracer
with tracer.start_as_current_span("my_operation") as span:
    answer = rag.rag(query)
    span.set_attribute("answer", answer)

{
    "name": "my_operation",
    "context": {
        "trace_id": "0xb0b95a4634b0b412561110b2e8758a64",
        "span_id": "0xd9684403e8384dda",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T02:40:06.613446Z",
    "end_time": "2026-07-20T02:40:10.210735Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "answer": "It keeps calling the model inside a `while True` loop.\n\nEach iteration:\n1. Send the full message history to the model.\n2. Check the response for any `function_call`.\n3. If there is one, run the tool, append the tool output to `messages`, and loop again.\n4. If there are no function calls, `break` out of the loop.\n\nSo the stop condition is:\n\n- **no function calls in the latest response**\n\nThat means the model has given its final answer, and the agent is done."
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "te

## Q1. First trace

Wrap the rag() method so each call produces a span. The simplest way is to create a RAGTraced subclass of RAGBase that wraps rag(), search(), and llm() each in their own span.

Run this query:

    How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary. Count the spans in the console output - each one is a separate ReadableSpan entry. How many spans does the trace produce?

- 1
- 3
- 5
- 7

Answer: 3

In [40]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [41]:
from dotenv import load_dotenv

load_dotenv()

from starter import index, client, RAGTraced

ragt = RAGTraced(tracer, index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = ragt.rag(query)

## Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you want to them with set_attribute. We already use spans to record how long each step takes. Now we'll add the metrics we care about: tokens and cost.

Read the token usage from the LLM response (the llm() method in the starter already returns the raw response object) and set them as attributes on the llm span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

- 700
- 7000
- 70000
- 700000

These numbers vary between runs. Pick the closest option.

The input tokens, output tokens, and total costs is returned by the 2nd span output as attributes. The input tokens consumed is 7,111.

Answer: 7000

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from dotenv import load_dotenv

load_dotenv()

from starter import index, client, RAGTraced

ragt = RAGTraced(tracer, index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = ragt.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x75fd99e0e01d0f4e45ef30caca204322",
        "span_id": "0x1a1a824d77975941",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb057d2467eab6b41",
    "start_time": "2026-07-20T07:16:23.995709Z",
    "end_time": "2026-07-20T07:16:23.996362Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "eb3ce461-aaf3-403e-b28d-047e65d78cd4",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x75fd99e0e01d0f4e45ef30caca204322",
        "span_id": "0xc9db4c41020b7a25",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q3. Span timing

Each span automatically records its duration. Look at the console output from Q1 and find the durations for the search span and the llm span.

For a typical query, roughly how long does the LLM call take?

- Under 100ms
- 100-500ms
- 500-2000ms
- Over 2000ms

The first call can be slower (cold start). Pick the range you see most often.

Answer: Over 2000ms

In [11]:
from datetime import datetime as dt

format_pattern = "%Y-%m-%dT%H:%M:%S.%f%z"
start1 = dt.strptime("2026-07-20T07:01:04.651894Z", format_pattern)
end1 = dt.strptime("2026-07-20T07:01:04.652520Z", format_pattern)

start2 = dt.strptime("2026-07-20T07:01:04.652972Z", format_pattern)
end2 = dt.strptime("2026-07-20T07:01:07.070726Z", format_pattern)

print(end1 - start1, end2 - start2)

0:00:00.000626 0:00:02.417754


## Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there. But we need to create a custom exporter. Instead of printing the spans, it will save them to the database.

OTel calls the exporter through the same span processor we already use, we just swap the destination.

Now we will create a custom exporter that saves each finished span to a SQLite database. The exporter extends SpanExporter. It has the following methods:

- export method that receives a list of ReadableSpan objects
- shutdown and force_flush methods

Let's implement it:

```python
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
```

Replace the console exporter with this new exporter:

```python
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```

Re-run the query from Q1. Which span names appear in the spans table?

- Only rag
- rag and llm
- rag, search, and llm
- search, llm, and judge

Answer: rag, search, and llm

In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [14]:
from dotenv import load_dotenv

load_dotenv()

from starter import index, client, RAGTraced

ragt = RAGTraced(tracer, index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = ragt.rag(query)

In [15]:
import sqlite3

connection = sqlite3.connect("traces.db")

try:
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM spans")
    rows = cursor.fetchall()
    for row in rows:
        print(row)

finally:
    connection.close()

('search', 1784534901907787000, 1784534901908424000, None, None, None)
('llm', 1784534901909205000, 1784534904401724000, 7111, 115, 0.00585075)
('rag', 1784534901907759000, 1784534904402605000, None, None, None)
('search', 1784535101675535000, 1784535101679046000, None, None, None)
('llm', 1784535101679922000, 1784535103894494000, 5676, 27, 0.0043785)
('rag', 1784535101675473000, 1784535103896746000, None, None, None)
('search', 1784537613159559000, 1784537613161801000, None, None, None)
('llm', 1784537613162833000, 1784537615739658000, 7111, 124, 0.005891250000000001)
('rag', 1784537613159507000, 1784537615741629000, None, None, None)
('search', 1784537780322912000, 1784537780323573000, None, None, None)
('llm', 1784537780324335000, 1784537782799482000, 7111, 100, 0.00578325)
('rag', 1784537780322884000, 1784537782800417000, None, None, None)
('search', 1784537899097736000, 1784537899099971000, None, None, None)
('llm', 1784537899101172000, 1784537901337445000, 7111, 117, 0.00585975)


## Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced RAG, then query the database.

The rag span wraps everything, so its duration includes both search and llm. To see where time actually goes, exclude the rag span and compare the children.

Using SQL (or pandas), compute the total duration for each span name excluding rag. Which span type takes the most total time?

- search
- llm
- They're all about the same

Answer: llm

In [5]:
query = "Can I still join the course?"
answer = ragt.rag(query)

In [29]:
connection = sqlite3.connect("traces.db")

try:
    cursor = connection.cursor()
    cursor.execute("""
    SELECT *
        , time((end_time - start_time) / 1000000000.0, 'unixepoch', 'subsec') AS duration
        -- , datetime(start_time / 1000000000.0, 'unixepoch', 'subsec') AS st_actual
    FROM spans 
    WHERE name NOT IN ('rag')
    """)
    rows = cursor.fetchall()
    for row in rows:
        print(row)

finally:
    connection.close()

('search', 1784534901907787000, 1784534901908424000, None, None, None, '00:00:00.001')
('llm', 1784534901909205000, 1784534904401724000, 7111, 115, 0.00585075, '00:00:02.493')
('search', 1784535101675535000, 1784535101679046000, None, None, None, '00:00:00.004')
('llm', 1784535101679922000, 1784535103894494000, 5676, 27, 0.0043785, '00:00:02.215')


In [36]:
connection = sqlite3.connect("traces.db")

try:
    cursor = connection.cursor()
    cursor.execute("""
    SELECT name
        , time(SUM((end_time - start_time) / 1000000000.0), 'unixepoch', 'subsec') AS total_duration
    FROM spans 
    WHERE name NOT IN ('rag')
    GROUP BY name
    """)
    rows = cursor.fetchall()
    for row in rows:
        print(row)

finally:
    connection.close()

# llm takes a lot longer than search.

('llm', '00:00:04.707')
('search', '00:00:00.004')


## Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you is how stable your system is. If the same query always produces the same number of input tokens, the context your RAG retrieves is consistent. If it varies a lot, something in the search may be unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls total in the database). Then compute the input tokens for each llm span.

How much do the input tokens vary across these 4 runs?

- They're identical
- Within 10% of each other
- Within 50% of each other
- They vary more than 50%

Answer: They're identical

In [18]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("traces.db")

try:
    query = "SELECT name, input_tokens FROM spans WHERE name IN ('llm')"
    
    df = pd.read_sql_query(query, connection)

finally:
    connection.close()

# View your data
print(df)

  name  input_tokens
0  llm          7111
1  llm          5676
2  llm          7111
3  llm          7111
4  llm          7111
5  llm          7111
6  llm          7111
